# V4 Transformer Adaptive Retrieval Training

> Self-contained Colab notebook. No GitHub clone is required.
>
> Paths assume Google Drive layout used throughout this project.

## Goal

Train V4, a Transformer sequence-aware Two-Tower retrieval model:

```text
latest user state + recent sequence -> adaptive user embedding
item features -> item embedding
dot product -> retrieval score
```

V4 keeps the V3 negative strategy: in-batch negatives + mixed popularity/random sampled negatives.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import importlib.util
import subprocess
import sys

required = ['pyarrow', 'torch', 'pandas', 'numpy']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages are available.')

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math
import random
import time
from typing import Iterable

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('torch:', torch.__version__)

In [ ]:
@dataclass
class V4Config:
    adaptive_root: str = '/content/drive/MyDrive/recsys/data/gold/adaptive/v1'
    base_gold_root: str = '/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab'
    output_dir: str = '/content/recsys_work/artifacts/adaptive/v4_sequence_retrieval'
    seed: int = 42
    smoke_mode: bool = False
    max_train_examples: int | None = None
    max_validation_examples: int | None = None
    max_item_pool_examples: int | None = None
    batch_size: int = 2048
    eval_batch_size: int = 2048
    epochs: int = 12
    learning_rate: float = 8e-4
    weight_decay: float = 1e-5
    retrieval_dim: int = 128
    user_hidden_dims: tuple[int, ...] = (768, 384)
    item_hidden_dims: tuple[int, ...] = (768, 384)
    transformer_dim: int = 256
    transformer_heads: int = 4
    transformer_layers: int = 2
    transformer_dropout: float = 0.1
    dropout: float = 0.12
    temperature: float = 0.05
    sequence_max_len: int = 30
    negative_sample_count: int = 8192
    popular_negative_fraction: float = 0.7
    item_popularity_column: str = 'item_hist_events'
    item_popularity_alpha: float = 0.75
    eval_top_ks: tuple[int, ...] = (10, 50, 100)
    candidate_eval_user_limit: int | None = None
    candidate_eval_item_limit: int | None = None
    early_stopping_patience: int = 3
    early_stopping_metric: str = 'ndcg@50'
    save_large_cache_to_drive: bool = False
    local_cache_dir: str = '/content/recsys_work/adaptive_cache/v4'

CONFIG = V4Config()
if CONFIG.smoke_mode:
    CONFIG.max_train_examples = 100_000
    CONFIG.max_validation_examples = 30_000
    CONFIG.max_item_pool_examples = 300_000
    CONFIG.batch_size = 512
    CONFIG.eval_batch_size = 512
    CONFIG.epochs = 1
    CONFIG.negative_sample_count = 1024
    CONFIG.candidate_eval_user_limit = 200
    CONFIG.candidate_eval_item_limit = 100_000

ADAPTIVE_ROOT = Path(CONFIG.adaptive_root)
BASE_GOLD_ROOT = Path(CONFIG.base_gold_root)
OUTPUT_DIR = Path(CONFIG.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_DIR = Path(CONFIG.local_cache_dir)
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(CONFIG), indent=2, default=str))

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parquet_files(path: str | Path) -> list[Path]:
    path = Path(path)
    files = sorted(p for p in path.rglob('*.parquet') if p.is_file() and not p.name.startswith('.'))
    if not files:
        visible = sorted(x.name for x in path.iterdir())[:30] if path.exists() and path.is_dir() else []
        raise FileNotFoundError(f'No parquet files found under {path}. Visible entries: {visible}')
    return files


def read_parquet(path: str | Path, columns=None, max_rows: int | None = None, filter_expr=None) -> pd.DataFrame:
    dataset = ds.dataset([str(p) for p in parquet_files(path)], format='parquet')
    if columns:
        missing = sorted(set(columns) - set(dataset.schema.names))
        if missing:
            raise ValueError(f'{path} missing columns: {missing}')
    if max_rows is None:
        return dataset.to_table(columns=columns, filter=filter_expr).to_pandas()
    return dataset.head(max_rows, columns=columns, filter=filter_expr).to_pandas()


def save_json(path: str | Path, payload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f:
        json.dump(payload, f, indent=2, sort_keys=True, default=str)


def now() -> str:
    return time.strftime('%Y-%m-%d %H:%M:%S')

set_seed(CONFIG.seed)

In [ ]:
required_paths = {
    'adaptive train': ADAPTIVE_ROOT / 'sequence_examples/train',
    'adaptive validation': ADAPTIVE_ROOT / 'sequence_examples/validation',
    'base vocabularies': BASE_GOLD_ROOT / 'vocabularies',
    'base numeric stats': BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
}
for name, path in required_paths.items():
    ok = path.exists()
    print(f'{name:24s}', ok, path)
    if not ok:
        raise FileNotFoundError(path)

In [ ]:
USER_CATEGORICAL = ('user_active_degree',)
ITEM_CATEGORICAL = ('video_type', 'upload_type')
USER_ID_FEATURES = ('user_id',)
ITEM_ID_FEATURES = ('video_id',)
USER_NUMERIC = (
    'is_lowactive_period','is_live_streamer','is_video_author','follow_user_num','fans_user_num','friend_user_num','register_days',
    *[f'onehot_feat{i}' for i in range(18)],
    'user_hist_events','user_hist_long_view_rate','user_hist_like_rate','user_hist_comment_rate','user_hist_forward_rate','user_hist_follow_rate',
    'user_hist_hate_rate','user_hist_avg_watch_ratio','user_hist_avg_play_time_sec','session_event_index','session_elapsed_sec',
    'session_prior_long_view_rate','session_prior_like_rate','session_prior_hate_rate','session_prior_avg_watch_ratio',
    'session_vs_user_long_view_delta','session_vs_user_watch_ratio_delta','event_hour','event_dayofweek','tab','is_rand'
)
ITEM_NUMERIC = (
    'video_duration_sec','aspect_ratio','visible_status','music_type','upload_age_days_at_event','item_hist_events',
    'item_hist_long_view_rate','item_hist_like_rate','item_hist_hate_rate','item_hist_avg_watch_ratio'
)
SEQUENCE_COLUMNS = (
    'recent_video_ids','recent_target_classes','recent_engagement_strengths','recent_long_views','recent_likes',
    'recent_hates','recent_clicks','recent_time_ms','recent_event_count','seconds_since_last_event'
)
TARGET_COLUMNS = ('target_class','is_positive','is_observed_negative','is_ambiguous','long_view','is_like','is_follow','is_hate','engagement_strength')
BASE_COLUMNS = ('example_id','user_id','video_id','as_of_time','current_time_ms')
INPUT_COLUMNS = list(dict.fromkeys([*BASE_COLUMNS, *TARGET_COLUMNS, *USER_ID_FEATURES, *USER_CATEGORICAL, *USER_NUMERIC, *ITEM_ID_FEATURES, *ITEM_CATEGORICAL, *ITEM_NUMERIC, *SEQUENCE_COLUMNS]))
print('input columns:', len(INPUT_COLUMNS))

In [ ]:
class Vocabulary:
    def __init__(self, name: str, values: Iterable, index: dict | None = None):
        self.name = name
        if index is not None:
            self.index = dict(index)
        else:
            clean = pd.Series(list(values)).dropna().drop_duplicates().tolist()
            try:
                clean = sorted(clean)
            except TypeError:
                clean = sorted(str(x) for x in clean)
            self.index = {value: i + 1 for i, value in enumerate(clean)}
        self.size = max(self.index.values(), default=0) + 1

    def encode_many(self, values) -> np.ndarray:
        return np.asarray([self.index.get(v, 0) for v in values], dtype=np.int64)


def load_parquet_vocabulary(path: Path, feature: str) -> Vocabulary:
    df = read_parquet(path)
    idx_col = f'{feature}_idx'
    if idx_col not in df.columns:
        raise ValueError(f'{path} missing {idx_col}')
    return Vocabulary(feature, [], dict(zip(df[feature].tolist(), df[idx_col].astype(int).tolist())))


class NumericalPreprocessor:
    def __init__(self, stats_path: Path):
        payload = json.loads(stats_path.read_text())
        self.stats = payload.get('features', payload)

    def transform(self, frame: pd.DataFrame, features: tuple[str, ...]) -> np.ndarray:
        out = []
        for feature in features:
            if feature not in frame.columns:
                raise ValueError(f'Missing numeric feature {feature}')
            stats = self.stats.get(feature)
            values = pd.to_numeric(frame[feature], errors='coerce').astype('float32')
            if stats is None:
                clean = np.nan_to_num(values.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            else:
                fill = float(stats.get('mean', 0.0) or 0.0)
                values = values.fillna(fill)
                lo = stats.get('p01', stats.get('min'))
                hi = stats.get('p99', stats.get('max'))
                if lo is not None and hi is not None:
                    values = values.clip(float(lo), float(hi))
                mean = float(stats.get('mean', 0.0) or 0.0)
                std = float(stats.get('stddev', 0.0) or 0.0)
                if std > 1e-6:
                    values = (values - mean) / std
                else:
                    values = values * 0.0
                clean = np.nan_to_num(values.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            out.append(clean)
        return np.stack(out, axis=1).astype('float32') if out else np.zeros((len(frame), 0), dtype='float32')


def build_vocabs(train_frame: pd.DataFrame, item_pool: pd.DataFrame) -> dict[str, Vocabulary]:
    return {
        'user_active_degree': load_parquet_vocabulary(BASE_GOLD_ROOT / 'vocabularies/user_active_degree', 'user_active_degree'),
        'video_type': load_parquet_vocabulary(BASE_GOLD_ROOT / 'vocabularies/video_type', 'video_type'),
        'upload_type': load_parquet_vocabulary(BASE_GOLD_ROOT / 'vocabularies/upload_type', 'upload_type'),
        'user_id': Vocabulary('user_id', train_frame['user_id'].tolist()),
        'video_id': Vocabulary('video_id', item_pool['video_id'].tolist()),
        'event_type': Vocabulary('event_type', ['AMBIGUOUS_WEAK', 'OBSERVED_NEGATIVE', 'STRONG_POSITIVE']),
    }

In [ ]:
def as_sequence(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return []


def as_of_time_ms(frame: pd.DataFrame) -> np.ndarray:
    return (pd.to_datetime(frame['as_of_time']).astype('int64') // 1_000_000).to_numpy(dtype='int64')


def sanitize_point_in_time_sequences(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop any sequence events that are not strictly before the target as_of_time.

    This is intentionally conservative and protects training/evaluation even if the
    adaptive data builder attached a same-second history row ambiguously.
    """
    frame = frame.copy()
    boundaries = as_of_time_ms(frame)
    sequence_cols = [
        'recent_video_ids', 'recent_target_classes', 'recent_engagement_strengths',
        'recent_long_views', 'recent_likes', 'recent_hates', 'recent_clicks', 'recent_time_ms',
    ]
    cleaned = {col: [] for col in sequence_cols if col in frame.columns}
    cleaned_counts = []
    dropped_rows = 0
    dropped_events = 0
    for (_, row), boundary in zip(frame.iterrows(), boundaries):
        times = as_sequence(row.get('recent_time_ms'))
        keep = [i for i, t in enumerate(times) if t is not None and not pd.isna(t) and float(t) < float(boundary)]
        if len(keep) != len(times):
            dropped_rows += 1
            dropped_events += len(times) - len(keep)
        for col in cleaned:
            values = as_sequence(row.get(col))
            if len(values) == len(times):
                cleaned[col].append([values[i] for i in keep])
            else:
                cleaned[col].append(values[-len(keep):] if keep else [])
        cleaned_counts.append(len(keep))
    for col, values in cleaned.items():
        frame[col] = values
    frame['recent_event_count'] = cleaned_counts
    frame['current_time_ms'] = boundaries
    print(f'point-in-time sanitize: rows_with_drops={dropped_rows:,} dropped_events={dropped_events:,}')
    return frame

def load_split(split: str, max_examples: int | None) -> pd.DataFrame:
    filt = ds.field('target_class') == 'STRONG_POSITIVE'
    frame = read_parquet(ADAPTIVE_ROOT / 'sequence_examples' / split, INPUT_COLUMNS, max_examples, filt)
    frame = frame.reset_index(drop=True)
    frame = sanitize_point_in_time_sequences(frame)
    print(split, frame.shape)
    return frame


def load_item_pool(split='train', max_examples: int | None = None) -> pd.DataFrame:
    cols = list(dict.fromkeys([*ITEM_ID_FEATURES, *ITEM_CATEGORICAL, *ITEM_NUMERIC]))
    frame = read_parquet(ADAPTIVE_ROOT / 'sequence_examples' / split, cols, max_examples)
    frame = frame.drop_duplicates('video_id', keep='last').reset_index(drop=True)
    print('item pool', split, frame.shape)
    return frame

train_frame = load_split('train', CONFIG.max_train_examples)
validation_frame = load_split('validation', CONFIG.max_validation_examples)
item_pool = load_item_pool('train', CONFIG.max_item_pool_examples)
validation_item_pool = load_item_pool('validation', CONFIG.candidate_eval_item_limit)

if train_frame.empty or validation_frame.empty:
    raise ValueError('No positive examples loaded. Check adaptive sequence_examples and target_class values.')

In [ ]:
numeric = NumericalPreprocessor(BASE_GOLD_ROOT / 'transforms/numeric_stats.json')
vocabs = build_vocabs(train_frame, item_pool)
print({k: v.size for k, v in vocabs.items()})
save_json(OUTPUT_DIR / 'vocab_summary.json', {k: v.size for k, v in vocabs.items()})

In [ ]:
def pad_numeric_sequence(values, max_len: int, dtype='float32'):
    if not isinstance(values, (list, tuple, np.ndarray)):
        values = []
    values = list(values)[-max_len:]
    out = np.zeros(max_len, dtype=dtype)
    if values:
        out[-len(values):] = np.asarray(values, dtype=dtype)
    return out


def pad_encoded_sequence(values, vocab: Vocabulary | None, max_len: int):
    if not isinstance(values, (list, tuple, np.ndarray)):
        values = []
    values = list(values)[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    if values:
        encoded = vocab.encode_many(values) if vocab is not None else np.asarray(values, dtype=np.int64)
        out[-len(values):] = encoded[-max_len:]
    return out


class AdaptiveDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        return self.frame.iloc[idx].to_dict()


class AdaptiveCollator:
    def __init__(self, vocabs, numeric, max_len: int):
        self.vocabs = vocabs
        self.numeric = numeric
        self.max_len = max_len
    def __call__(self, rows):
        frame = pd.DataFrame(rows)
        return encode_frame(frame, self.vocabs, self.numeric, self.max_len)


def encode_frame(frame: pd.DataFrame, vocabs, numeric, max_len: int) -> dict[str, torch.Tensor]:
    batch = {}
    for feature in USER_ID_FEATURES + USER_CATEGORICAL + ITEM_ID_FEATURES + ITEM_CATEGORICAL:
        batch[feature] = torch.as_tensor(vocabs[feature].encode_many(frame[feature].tolist()), dtype=torch.long)
    batch['user_numeric'] = torch.as_tensor(numeric.transform(frame, USER_NUMERIC), dtype=torch.float32)
    batch['item_numeric'] = torch.as_tensor(numeric.transform(frame, ITEM_NUMERIC), dtype=torch.float32)
    batch['seq_video_id'] = torch.as_tensor(np.stack([pad_encoded_sequence(v, vocabs['video_id'], max_len) for v in frame['recent_video_ids']]), dtype=torch.long)
    batch['seq_event_type'] = torch.as_tensor(np.stack([pad_encoded_sequence(v, vocabs['event_type'], max_len) for v in frame['recent_target_classes']]), dtype=torch.long)
    engagement = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_engagement_strengths']])
    long_view = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_long_views']])
    likes = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_likes']])
    hates = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_hates']])
    clicks = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_clicks']])
    seq_signals = np.stack([engagement, long_view, likes, hates, clicks], axis=-1).astype('float32')
    batch['seq_signals'] = torch.as_tensor(seq_signals, dtype=torch.float32)
    current_ms = pd.to_numeric(frame['current_time_ms'], errors='coerce').fillna(0).to_numpy(dtype='float64')
    recent_ms = np.stack([pad_numeric_sequence(v, max_len, dtype='float64') for v in frame['recent_time_ms']])
    age_sec = np.maximum((current_ms[:, None] - recent_ms) / 1000.0, 0.0)
    age_sec[recent_ms <= 0] = 0.0
    age_bucket = np.clip(np.floor(np.log1p(age_sec) / np.log(2.0)), 0, 31).astype('int64')
    batch['seq_time_bucket'] = torch.as_tensor(age_bucket, dtype=torch.long)
    batch['seq_padding_mask'] = batch['seq_video_id'].eq(0)
    batch['label'] = torch.arange(len(frame), dtype=torch.long)
    batch['video_id_raw'] = torch.as_tensor(pd.to_numeric(frame['video_id'], errors='coerce').fillna(0).to_numpy(dtype='int64'))
    return batch


def move_batch(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

In [ ]:
def mlp(input_dim, hidden_dims, output_dim, dropout):
    layers = []
    prev = input_dim
    for hidden in hidden_dims:
        layers += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(dropout)]
        prev = hidden
    layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)


class V4TransformerRetrieval(nn.Module):
    def __init__(self, vocab_sizes: dict[str, int], config: V4Config):
        super().__init__()
        self.config = config
        self.user_id_emb = nn.Embedding(vocab_sizes['user_id'], 8, padding_idx=0)
        self.user_active_emb = nn.Embedding(vocab_sizes['user_active_degree'], 4, padding_idx=0)
        self.video_id_emb = nn.Embedding(vocab_sizes['video_id'], 16, padding_idx=0)
        self.video_type_emb = nn.Embedding(vocab_sizes['video_type'], 3, padding_idx=0)
        self.upload_type_emb = nn.Embedding(vocab_sizes['upload_type'], 8, padding_idx=0)
        self.seq_video_emb = nn.Embedding(vocab_sizes['video_id'], config.transformer_dim, padding_idx=0)
        self.seq_event_emb = nn.Embedding(vocab_sizes['event_type'], 32, padding_idx=0)
        self.seq_event_proj = nn.Linear(32, config.transformer_dim)
        self.seq_signal_proj = nn.Linear(5, config.transformer_dim)
        self.seq_time_emb = nn.Embedding(32, config.transformer_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=config.transformer_dim,
            nhead=config.transformer_heads,
            dim_feedforward=config.transformer_dim * 4,
            dropout=config.transformer_dropout,
            batch_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=config.transformer_layers)
        self.short_proj = nn.Sequential(nn.LayerNorm(config.transformer_dim), nn.Linear(config.transformer_dim, config.retrieval_dim), nn.ReLU())
        user_input = 8 + 4 + len(USER_NUMERIC)
        item_input = 16 + 3 + 8 + len(ITEM_NUMERIC)
        self.long_user = mlp(user_input, config.user_hidden_dims, config.retrieval_dim, config.dropout)
        self.item_tower = mlp(item_input, config.item_hidden_dims, config.retrieval_dim, config.dropout)
        self.gate = nn.Sequential(nn.Linear(config.retrieval_dim * 2, config.retrieval_dim), nn.Sigmoid())
        self.temperature = config.temperature

    def encode_sequence(self, batch):
        token = self.seq_video_emb(batch['seq_video_id'])
        token = token + self.seq_event_proj(self.seq_event_emb(batch['seq_event_type']))
        token = token + self.seq_signal_proj(batch['seq_signals'])
        token = token + self.seq_time_emb(batch['seq_time_bucket'].clamp(0, 31))
        mask = batch['seq_padding_mask']
        encoded = self.transformer(token, src_key_padding_mask=mask)
        valid = (~mask).float().unsqueeze(-1)
        pooled = (encoded * valid).sum(1) / valid.sum(1).clamp_min(1.0)
        return self.short_proj(pooled)

    def encode_user(self, batch):
        long_features = torch.cat([
            self.user_id_emb(batch['user_id']),
            self.user_active_emb(batch['user_active_degree']),
            batch['user_numeric'],
        ], dim=1)
        long_repr = self.long_user(long_features)
        short_repr = self.encode_sequence(batch)
        gate = self.gate(torch.cat([long_repr, short_repr], dim=1))
        fused = gate * short_repr + (1.0 - gate) * long_repr
        return F.normalize(fused, dim=1)

    def encode_item(self, batch):
        item_features = torch.cat([
            self.video_id_emb(batch['video_id']),
            self.video_type_emb(batch['video_type']),
            self.upload_type_emb(batch['upload_type']),
            batch['item_numeric'],
        ], dim=1)
        return F.normalize(self.item_tower(item_features), dim=1)

    def forward(self, batch, negative_batch=None):
        user_emb = self.encode_user(batch)
        pos_item_emb = self.encode_item(batch)
        item_embs = [pos_item_emb]
        if negative_batch is not None:
            item_embs.append(self.encode_item(negative_batch))
        all_item_emb = torch.cat(item_embs, dim=0)
        logits = user_emb @ all_item_emb.T / self.temperature
        return logits, user_emb, pos_item_emb

In [ ]:
# Point-in-time check: every sequence timestamp must be strictly earlier than target as_of_time.
def leakage_check(frame: pd.DataFrame, max_rows: int = 200_000):
    sample = frame.head(max_rows)
    boundaries = as_of_time_ms(sample)
    violations = 0
    checked = 0
    for (_, row), boundary in zip(sample.iterrows(), boundaries):
        times = as_sequence(row.get('recent_time_ms'))
        for t in times:
            if t is None or pd.isna(t):
                continue
            checked += 1
            if float(t) >= float(boundary):
                violations += 1
                break
    print('sequence timestamps checked:', checked, 'violating rows:', violations)
    if violations:
        raise AssertionError(f'Point-in-time leakage detected in {violations} rows')

leakage_check(train_frame)
leakage_check(validation_frame)

In [ ]:
vocab_sizes = {k: v.size for k, v in vocabs.items()}
model = V4TransformerRetrieval(vocab_sizes, CONFIG).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
loss_fn = nn.CrossEntropyLoss()
collator = AdaptiveCollator(vocabs, numeric, CONFIG.sequence_max_len)
train_loader = DataLoader(AdaptiveDataset(train_frame), batch_size=CONFIG.batch_size, shuffle=True, collate_fn=collator, num_workers=0)
validation_loader = DataLoader(AdaptiveDataset(validation_frame), batch_size=CONFIG.eval_batch_size, shuffle=False, collate_fn=collator, num_workers=0)
print('params:', sum(p.numel() for p in model.parameters()))

In [ ]:
negative_rng = np.random.default_rng(CONFIG.seed)
popularity = pd.to_numeric(item_pool.get(CONFIG.item_popularity_column, pd.Series(np.ones(len(item_pool)))), errors='coerce').fillna(0).to_numpy('float64')
weights = np.power(np.maximum(popularity, 0.0) + 1.0, CONFIG.item_popularity_alpha)
popularity_probs = weights / weights.sum() if weights.sum() > 0 else np.full(len(item_pool), 1 / len(item_pool))


def sample_negative_frame(count: int) -> pd.DataFrame:
    popular_count = int(round(count * CONFIG.popular_negative_fraction))
    random_count = count - popular_count
    parts = []
    if popular_count:
        parts.append(negative_rng.choice(len(item_pool), size=popular_count, replace=True, p=popularity_probs))
    if random_count:
        parts.append(negative_rng.integers(0, len(item_pool), size=random_count))
    idx = np.concatenate(parts) if parts else np.array([], dtype=np.int64)
    return item_pool.iloc[idx].reset_index(drop=True)


def make_negative_batch(count: int):
    neg_frame = sample_negative_frame(count)
    encoded = encode_frame(neg_frame.assign(**minimal_user_stub(len(neg_frame))), vocabs, numeric, CONFIG.sequence_max_len)
    return move_batch(encoded, DEVICE)


def minimal_user_stub(n: int) -> dict:
    stub = {c: 0 for c in USER_NUMERIC}
    stub.update({
        'user_id': train_frame['user_id'].iloc[0],
        'user_active_degree': train_frame['user_active_degree'].iloc[0],
        'recent_video_ids': [[] for _ in range(n)],
        'recent_target_classes': [[] for _ in range(n)],
        'recent_engagement_strengths': [[] for _ in range(n)],
        'recent_long_views': [[] for _ in range(n)],
        'recent_likes': [[] for _ in range(n)],
        'recent_hates': [[] for _ in range(n)],
        'recent_clicks': [[] for _ in range(n)],
        'recent_time_ms': [[] for _ in range(n)],
        'current_time_ms': 0,
    })
    return stub

In [ ]:
def evaluate_in_batch(loader, max_batches: int | None = 50):
    model.eval()
    totals = {'loss': 0.0, 'batches': 0}
    ks = CONFIG.eval_top_ks
    hits = {k: 0 for k in ks}
    ndcg = {k: 0.0 for k in ks}
    with torch.no_grad():
        for i, raw in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            batch = move_batch(raw, DEVICE)
            logits, _, _ = model(batch)
            labels = torch.arange(logits.shape[0], device=DEVICE)
            loss = loss_fn(logits, labels)
            ranks = torch.argsort(logits, dim=1, descending=True)
            for k in ks:
                top = ranks[:, :k]
                match = top.eq(labels[:, None])
                hits[k] += match.any(dim=1).sum().item()
                pos = match.float().argmax(dim=1) + 1
                found = match.any(dim=1)
                ndcg[k] += (found.float() / torch.log2(pos.float() + 1.0)).sum().item()
            totals['loss'] += loss.item()
            totals['batches'] += 1
    n = totals['batches'] * CONFIG.eval_batch_size
    out = {'loss': totals['loss'] / max(totals['batches'], 1)}
    for k in ks:
        out[f'hitrate@{k}'] = hits[k] / max(n, 1)
        out[f'ndcg@{k}'] = ndcg[k] / max(n, 1)
    return out

history = []
best_metric = -1.0
best_epoch = None
bad_epochs = 0
for epoch in range(1, CONFIG.epochs + 1):
    model.train()
    running = 0.0
    seen = 0
    start = time.time()
    for step, raw in enumerate(train_loader, start=1):
        batch = move_batch(raw, DEVICE)
        neg = make_negative_batch(CONFIG.negative_sample_count) if CONFIG.negative_sample_count > 0 else None
        logits, _, _ = model(batch, neg)
        labels = torch.arange(logits.shape[0], device=DEVICE)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running += loss.item() * logits.shape[0]
        seen += logits.shape[0]
        if step % 50 == 0:
            print(f'epoch={epoch} step={step} train_loss={running/max(seen,1):.4f}', flush=True)
    val = evaluate_in_batch(validation_loader)
    row = {'epoch': epoch, 'train_loss': running / max(seen, 1), **{f'validation_{k}': v for k, v in val.items()}, 'elapsed_min': (time.time() - start) / 60}
    history.append(row)
    print(row, flush=True)
    metric = val.get(CONFIG.early_stopping_metric, val.get('ndcg@50', -1.0))
    if metric > best_metric:
        best_metric = metric
        best_epoch = epoch
        bad_epochs = 0
        torch.save({'model_state_dict': model.state_dict(), 'config': asdict(CONFIG), 'vocab_sizes': vocab_sizes}, OUTPUT_DIR / 'best_model.pt')
    else:
        bad_epochs += 1
        if bad_epochs >= CONFIG.early_stopping_patience:
            print('early stopping')
            break

pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
print('best_epoch:', best_epoch, 'best_metric:', best_metric)

In [ ]:
def encode_items(frame: pd.DataFrame, batch_size=8192):
    embs = []
    ids = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(frame), batch_size):
            part = frame.iloc[start:start+batch_size].reset_index(drop=True)
            enc = encode_frame(part.assign(**minimal_user_stub(len(part))), vocabs, numeric, CONFIG.sequence_max_len)
            emb = model.encode_item(move_batch(enc, DEVICE)).detach().cpu().numpy()
            embs.append(emb)
            ids.append(part['video_id'].to_numpy('int64'))
    return np.concatenate(ids), np.vstack(embs).astype('float32')


def encode_users(frame: pd.DataFrame, batch_size=2048):
    embs = []
    uids = []
    vids = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(frame), batch_size):
            part = frame.iloc[start:start+batch_size].reset_index(drop=True)
            enc = encode_frame(part, vocabs, numeric, CONFIG.sequence_max_len)
            emb = model.encode_user(move_batch(enc, DEVICE)).detach().cpu().numpy()
            embs.append(emb)
            uids.append(part['user_id'].to_numpy('int64'))
            vids.append(part['video_id'].to_numpy('int64'))
    return np.concatenate(uids), np.concatenate(vids), np.vstack(embs).astype('float32')

ckpt = torch.load(OUTPUT_DIR / 'best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
query_frame = validation_frame.sort_values('as_of_time').groupby('user_id').tail(1).reset_index(drop=True)
if CONFIG.candidate_eval_user_limit:
    query_frame = query_frame.head(CONFIG.candidate_eval_user_limit)
if CONFIG.candidate_eval_item_limit:
    validation_item_pool_eval = validation_item_pool.head(CONFIG.candidate_eval_item_limit)
else:
    validation_item_pool_eval = validation_item_pool
print('candidate eval queries/items:', query_frame.shape, validation_item_pool_eval.shape)

candidate_ids, item_embeddings = encode_items(validation_item_pool_eval)
query_user_ids, query_true_items, user_embeddings = encode_users(query_frame)
scores = user_embeddings @ item_embeddings.T
max_k = max(CONFIG.eval_top_ks)
top_idx = np.argpartition(-scores, kth=min(max_k, scores.shape[1]-1), axis=1)[:, :max_k]
row_scores = np.take_along_axis(scores, top_idx, axis=1)
order = np.argsort(-row_scores, axis=1)
top_idx = np.take_along_axis(top_idx, order, axis=1)

def exact_next_metrics(candidate_ids, top_idx, query_true_items, top_ks):
    out = {}
    for k in top_ks:
        recs = candidate_ids[top_idx[:, :k]]
        hit = (recs == query_true_items[:, None]).any(axis=1)
        out[f'hitrate@{k}'] = float(hit.mean())
        ndcg_vals = []
        for row, true_item in zip(recs, query_true_items):
            pos = np.where(row == true_item)[0]
            ndcg_vals.append(0.0 if len(pos) == 0 else 1.0 / math.log2(int(pos[0]) + 2))
        out[f'ndcg@{k}'] = float(np.mean(ndcg_vals))
    out['evaluated_users'] = int(len(query_true_items))
    out['candidate_items'] = int(len(candidate_ids))
    return out


def multi_positive_metrics(candidate_ids, top_idx, query_user_ids, validation_positive_frame, top_ks):
    relevance = (
        validation_positive_frame.groupby('user_id')['video_id']
        .apply(lambda s: set(int(x) for x in s.dropna().tolist()))
        .to_dict()
    )
    rows = []
    for row_idx, user_id in enumerate(query_user_ids):
        relevant = relevance.get(int(user_id), set())
        if not relevant:
            continue
        ranked = candidate_ids[top_idx[row_idx, :max(top_ks)]]
        rows.append((ranked, relevant))
    out = {}
    for k in top_ks:
        hits = []
        recalls = []
        ndcgs = []
        for ranked, relevant in rows:
            top = [int(x) for x in ranked[:k]]
            rel_flags = [1 if item in relevant else 0 for item in top]
            hit_count = sum(rel_flags)
            hits.append(hit_count > 0)
            recalls.append(hit_count / max(len(relevant), 1))
            dcg = sum(flag / math.log2(pos + 2) for pos, flag in enumerate(rel_flags))
            ideal_hits = min(len(relevant), k)
            idcg = sum(1.0 / math.log2(pos + 2) for pos in range(ideal_hits))
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
        out[f'hitrate@{k}'] = float(np.mean(hits)) if hits else 0.0
        out[f'recall@{k}'] = float(np.mean(recalls)) if recalls else 0.0
        out[f'ndcg@{k}'] = float(np.mean(ndcgs)) if ndcgs else 0.0
    out['evaluated_users'] = int(len(rows))
    out['candidate_items'] = int(len(candidate_ids))
    return out

exact_next = exact_next_metrics(candidate_ids, top_idx, query_true_items, CONFIG.eval_top_ks)
multi_positive = multi_positive_metrics(candidate_ids, top_idx, query_user_ids, validation_frame, CONFIG.eval_top_ks)
metrics = {
    'exact_next_item_validation': exact_next,
    'multi_positive_candidate_validation': multi_positive,
}
print(json.dumps(metrics, indent=2))
save_json(OUTPUT_DIR / 'candidate_validation.json', metrics)
if CONFIG.save_large_cache_to_drive:
    np.savez_compressed(OUTPUT_DIR / 'validation_item_embedding_cache.npz', video_id=candidate_ids, embedding=item_embeddings)
else:
    np.savez_compressed(LOCAL_CACHE_DIR / 'validation_item_embedding_cache.npz', video_id=candidate_ids, embedding=item_embeddings)
    print('large embedding cache kept off Drive:', LOCAL_CACHE_DIR / 'validation_item_embedding_cache.npz')
save_json(OUTPUT_DIR / 'metrics.json', {'best_epoch': best_epoch, 'best_metric': best_metric, 'candidate_validation': metrics, 'history': history})
save_json(OUTPUT_DIR / 'config.json', asdict(CONFIG))

## Download Local V4 Artifacts

Artifacts are written to Colab local disk. Run this cell after training to download them to your machine.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/v4_sequence_retrieval', 'zip', str(OUTPUT_DIR))
print('created', zip_path)
files.download(zip_path)